# 2단계 ⓒ — 트리 기준선 · 스태킹 · 종합 (22_stacking_summary)
21 의 GNN 6종 옆에 **같은 피처의 트리**(XGBoost · LightGBM, 30열, 시드 3) 를 놓고, **GNN + 트리 스태킹**이 둘 중 하나보다 나은지 본다.
그 다음 전부 한 표로 모아 3단계(Large) 에 올릴 상위 2개를 고른다.

스태킹 두 가지 (학습 구간 점수는 과적합돼 있어 **검증 표본에서만** 결합 규칙을 정하고 테스트에 적용):
- `blend_<kind>`: 순위 평균 — `w · rank(gnn) + (1−w) · rank(xgb)`, w 는 검증 AP 최대 (0 ~ 1, 0.1 간격)
- `stack_<kind>`: 로지스틱 회귀 — 입력 [logit(xgb), logit(gnn)], 검증 표본(가중) 으로 적합

출력은 21 과 같은 폴더 `results/HI-Small/model_compare/` 에 같은 파일 형식 (`<name>_results.json`, 점수 parquet, 패턴/차수/컷 csv) → 종합표가 한 번에 읽는다

## 0. 작업 준비

In [ ]:
import sys, json, time, gc
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

sys.path.append(str(Path.cwd()))
import bc_lib as B
L = B.L
import bc_models as M

In [ ]:
RUN_NAME = 'stage2'
cfg = B.Config(dataset='HI-Small', smoke=False, val_neg_rate=0.05)
OUT_DIR = cfg.out_dir('model_compare')
meta = json.loads((cfg.feat_dir / f'{RUN_NAME}_meta.json').read_text(encoding='utf-8'))
GNN_FEATURES, TREE_FEATURES, TREE_COLS = meta['gnn_features'], meta['tree_features'], meta['tree_cols']
FEAT_PATH = Path(meta['feat_path'])
MODELS = ['gine', 'gine128', 'pna', 'gat', 'trans', 'salt_trans']
SEEDS  = [0, 1, 2]
TREES  = ['xgboost', 'lightgbm']
len(TREE_FEATURES), TREE_COLS[:5]

## 1. 데이터 — 20 에서 만든 행렬·표본

In [ ]:
con = L.connect(threads=8, memory='8GB')
currency_codes, format_codes = B.code_tables(con, cfg)
tx = B.load_tx_lowmem(con, cfg)
feat_z = np.load(FEAT_PATH, mmap_mode='r'); assert feat_z.shape == (len(tx), len(GNN_FEATURES))
val_df  = pd.read_parquet(meta['samples']['val']); test_df = pd.read_parquet(meta['samples']['test'])
in_degree = pd.read_parquet(meta['samples']['indegree']).set_index('node_id')['in_degree']
con.close()
all_edge_ids = tx['edge_id'].to_numpy()
val_idx = B.rows_of(all_edge_ids, val_df['edge_id'].to_numpy()); test_idx = B.rows_of(all_edge_ids, test_df['edge_id'].to_numpy())
in_train = B.window_mask(tx, cfg.train_window); train_rows = np.flatnonzero(in_train)
y_train = tx['y'].to_numpy()[in_train]; y_val, w_val = val_df['y'].to_numpy(), val_df['w'].to_numpy()
del all_edge_ids, tx; gc.collect()
X_train = B.rows_matrix(feat_z, train_rows, TREE_COLS); X_val = B.rows_matrix(feat_z, val_idx, TREE_COLS); X_test = B.rows_matrix(feat_z, test_idx, TREE_COLS)
print(X_train.shape, X_val.shape, X_test.shape, f'{(X_train.nbytes + X_test.nbytes) / 1e9:.2f} GB')

## 2. 공통 — 점수 → 결과 파일 (21 과 같은 형식)

In [ ]:
def save_run(name, kind, seed, family, val_score, test_score, extra=None):
    """검증·테스트 점수를 받아 21 과 같은 파일 묶음을 쓴다. 돌려주는 것: 테스트 지표"""
    pd.DataFrame(dict(edge_id=val_df.edge_id, y=val_df.y, w=val_df.w, score=val_score)).to_parquet(OUT_DIR / f'{name}_val_scores.parquet', index=False)
    sc = pd.DataFrame(dict(edge_id=test_df.edge_id, y=test_df.y, score=test_score)); sc.to_parquet(OUT_DIR / f'{name}_test_scores.parquet', index=False)
    B.pattern_recall(sc).to_csv(OUT_DIR / f'{name}_pattern.csv', encoding='utf-8-sig')
    B.degree_stratified(sc, test_df['dst'].to_numpy(), in_degree).to_csv(OUT_DIR / f'{name}_degree.csv', encoding='utf-8-sig')
    B.cut_table(sc).to_csv(OUT_DIR / f'{name}_cuts.csv', encoding='utf-8-sig')
    rec = dict(name=name, kind=kind, seed=seed, family=family, val=B.evaluate_sample(val_df, val_score), test=B.evaluate_sample(test_df, test_score), **(extra or {}))
    (OUT_DIR / f'{name}_results.json').write_text(json.dumps(rec, ensure_ascii=False, indent=2, default=float), encoding='utf-8')
    return rec['test']

def done(name):
    return (OUT_DIR / f'{name}_results.json').exists()

## 3. 트리 기준선 — XGBoost · LightGBM × 시드 3 (30 피처, 1단계와 같은 설정)

In [ ]:
import dataclasses
for seed in SEEDS:
    if all(done(f'{t}_s{seed}') for t in TREES):
        print(f'[trees s{seed}] 이미 완료 — 건너뜀'); continue
    cfg_t = dataclasses.replace(cfg, seed=seed); results = {}
    t0 = time.time()
    lgb_model, xgb_model = B.train_trees(X_train, y_train, X_val, y_val, w_val, X_test, val_df, test_df, cfg_t, results)
    for tname, model in (('xgboost', xgb_model), ('lightgbm', lgb_model)):
        m = save_run(f'{tname}_s{seed}', tname, seed, 'tree', model.predict_proba(X_val)[:, 1], model.predict_proba(X_test)[:, 1],
                     extra=dict(best_iteration=results[tname]['best_iteration'], train_sec=results[tname]['train_sec'], n_features=len(TREE_FEATURES)))
        print(f"  {tname}_s{seed}  test AP {m['AP']:.4f}  recall@1% {m['recall_at_1pct']:.4f}  unmatched@1% {m['unmatched_recall_at_1pct']:.4f}")
    imp = pd.DataFrame({'feature': TREE_FEATURES, 'xgboost_gain': [xgb_model.get_booster().get_score(importance_type='gain').get(f'f{i}', 0.0) for i in range(len(TREE_FEATURES))],
                        'lightgbm_gain': lgb_model.booster_.feature_importance('gain')})
    imp.sort_values('xgboost_gain', ascending=False).to_csv(OUT_DIR / f'importance_s{seed}.csv', index=False, encoding='utf-8-sig')
    B.log(f'[trees s{seed}] 끝 {(time.time()-t0)/60:.0f}분')
pd.read_csv(OUT_DIR / 'importance_s0.csv').head(12)

## 4. 스태킹 — GNN(kind, seed) × XGBoost(같은 seed)
GNN 검증 점수는 21 이 저장한 `<kind>_s<seed>_val_scores.parquet`. 21 이 아직 안 끝난 조합은 건너뛴다 (나중에 다시 실행하면 채워짐)

In [ ]:
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression

def ranks(s):
    return rankdata(s) / len(s)

def logit(p, eps=1e-6):
    p = np.clip(p.astype(np.float64), eps, 1 - eps); return np.log(p / (1 - p))

def load_pair(kind, seed):
    g_val = pd.read_parquet(OUT_DIR / f'{kind}_s{seed}_val_scores.parquet'); g_test = pd.read_parquet(OUT_DIR / f'{kind}_s{seed}_test_scores.parquet')
    x_val = pd.read_parquet(OUT_DIR / f'xgboost_s{seed}_val_scores.parquet'); x_test = pd.read_parquet(OUT_DIR / f'xgboost_s{seed}_test_scores.parquet')
    assert (g_val.edge_id.to_numpy() == x_val.edge_id.to_numpy()).all() and (g_test.edge_id.to_numpy() == x_test.edge_id.to_numpy()).all()
    return g_val.score.to_numpy(), g_test.score.to_numpy(), x_val.score.to_numpy(), x_test.score.to_numpy()

def stack_one(kind, seed):
    if not (OUT_DIR / f'{kind}_s{seed}_val_scores.parquet').exists() or not done(f'xgboost_s{seed}'):
        print(f'[{kind}_s{seed}] GNN 또는 XGB 점수 없음 — 건너뜀'); return
    if done(f'blend_{kind}_s{seed}') and done(f'stack_{kind}_s{seed}'):
        return
    gv, gt, xv, xt = load_pair(kind, seed)
    # (a) 순위 평균 — w 는 검증 AP 최대
    rgv, rxv = ranks(gv), ranks(xv); best = max(((B.evaluate_sample(val_df, w * rgv + (1 - w) * rxv)['AP'], w) for w in np.round(np.arange(0, 1.01, 0.1), 1)))
    w = best[1]; rgt, rxt = ranks(gt), ranks(xt)
    m = save_run(f'blend_{kind}_s{seed}', f'blend_{kind}', seed, 'stack', w * rgv + (1 - w) * rxv, w * rgt + (1 - w) * rxt, extra=dict(weight_gnn=w, base=[kind, 'xgboost']))
    # (b) 로지스틱 회귀 — [logit(xgb), logit(gnn)], 검증 표본 가중
    lr = LogisticRegression(C=1.0, max_iter=1000).fit(np.c_[logit(xv), logit(gv)], y_val, sample_weight=w_val)
    m2 = save_run(f'stack_{kind}_s{seed}', f'stack_{kind}', seed, 'stack', lr.predict_proba(np.c_[logit(xv), logit(gv)])[:, 1], lr.predict_proba(np.c_[logit(xt), logit(gt)])[:, 1],
                  extra=dict(coef=lr.coef_[0].tolist(), intercept=float(lr.intercept_[0]), base=[kind, 'xgboost']))
    print(f"[{kind}_s{seed}] blend w_gnn={w:.1f} test AP {m['AP']:.4f} | stack coef(xgb, gnn)={np.round(lr.coef_[0], 2)} test AP {m2['AP']:.4f}")

for seed in SEEDS:
    for kind in MODELS:
        stack_one(kind, seed)

## 5. 종합표 — GNN 6 · 트리 2 · 스태킹 12, 시드 평균 ± 표준편차

In [ ]:
recs = [json.loads(p.read_text(encoding='utf-8')) for p in sorted(OUT_DIR.glob('*_results.json'))]
rows = []
for r in recs:
    d = pd.read_csv(OUT_DIR / f"{r['name']}_degree.csv", index_col=0)
    rows.append(dict(kind=r['kind'], family=r['family'], seed=r['seed'], val_AP=r['val']['AP'], test_AP=r['test']['AP'], F1=r['test']['F1'], F1_at_0_5=r['test']['F1_at_0.5'],
                     recall_1pct=r['test']['recall_at_1pct'], unmatched_1pct=r['test']['unmatched_recall_at_1pct'],
                     hub50_F1=d.loc['[50-99]', 'F1'] if '[50-99]' in d.index else np.nan, hub100_F1=d.loc['[100+]', 'F1'] if '[100+]' in d.index else np.nan,
                     hub50_AP=d.loc['[50-99]', 'AP'] if '[50-99]' in d.index else np.nan, hub100_AP=d.loc['[100+]', 'AP'] if '[100+]' in d.index else np.nan))
runs = pd.DataFrame(rows); runs.to_csv(OUT_DIR / 'runs_all.csv', index=False, encoding='utf-8-sig')
METRICS = ['test_AP', 'F1', 'recall_1pct', 'unmatched_1pct', 'hub50_F1', 'hub100_F1']
g = runs.groupby(['family', 'kind'])
mean = g[METRICS].mean(); std = g[METRICS].std().fillna(0)
summary = (mean.round(3).astype(str) + ' ± ' + std.round(3).astype(str)); summary['n_seeds'] = g.size()
order = [('gnn', k) for k in MODELS] + [('tree', t) for t in TREES] + [('stack', f'blend_{k}') for k in MODELS] + [('stack', f'stack_{k}') for k in MODELS]
summary = summary.reindex([o for o in order if o in summary.index]); mean = mean.reindex(summary.index)
summary.to_csv(OUT_DIR / 'summary_all.csv', encoding='utf-8-sig'); summary

In [ ]:
# 그림 — 전체 AP vs 허브 F1 ([50-99]) 산점도. 오른쪽 위가 좋은 모델
fig, ax = plt.subplots(figsize=(6.5, 4.5))
colors = {'gnn': 'steelblue', 'tree': 'darkorange', 'stack': 'seagreen'}
for (fam, kind), r in mean.iterrows():
    ax.scatter(r.test_AP, r.hub50_F1, c=colors[fam], s=40); ax.annotate(kind, (r.test_AP, r.hub50_F1), fontsize=7, xytext=(3, 3), textcoords='offset points')
ax.set_xlabel('test AP (overall)'); ax.set_ylabel('F1  in-degree [50-99]'); ax.grid(alpha=.3)
ax.legend(handles=[plt.Line2D([], [], marker='o', ls='', c=c, label=f) for f, c in colors.items()], fontsize=8); plt.tight_layout()
plt.savefig(OUT_DIR / 'ap_vs_hub_f1.png', dpi=120); plt.show()

In [ ]:
# 선정 — 전체 AP 순위와 허브 F1([50-99] 와 [100+] 평균) 순위의 평균이 가장 좋은 2개 (GNN·스태킹 중에서. 트리는 비교 기준)
cand = mean.drop(index='tree', level='family', errors='ignore').copy()
cand['hub_F1'] = cand[['hub50_F1', 'hub100_F1']].mean(axis=1)
cand['rank_AP'] = cand['test_AP'].rank(ascending=False); cand['rank_hub'] = cand['hub_F1'].rank(ascending=False); cand['rank_mean'] = cand[['rank_AP', 'rank_hub']].mean(axis=1)
cand = cand.sort_values('rank_mean'); cand[['test_AP', 'hub_F1', 'rank_AP', 'rank_hub', 'rank_mean']].round(3)

In [ ]:
# GNN 이 트리보다 뭘 더 주나 — 패턴별 recall@1% 차이 (GNN 평균 − XGB 평균). 양수 = GNN 이 더 잡는 패턴
pat = pd.concat([pd.read_csv(OUT_DIR / f"{r['name']}_pattern.csv", index_col=0).assign(kind=r['kind']) for r in recs if r['family'] in ('gnn', 'tree')])
order_p = [p for p in B.PATTERN_ORDER + ['ALL'] if p in pat.index]
m = pat.reset_index().groupby(['kind', 'pattern_type'])['r@1%'].mean().unstack('pattern_type')[order_p]
diff = m.loc[[k for k in MODELS if k in m.index]] - m.loc['xgboost']
B.plot_heatmap(diff, title='pattern recall@1%: GNN − XGBoost (mean of seeds)', path=OUT_DIR / 'pattern_gnn_minus_xgb.png', fmt='{:+.2f}', vmax=0.3)
m.loc[[k for k in MODELS + TREES if k in m.index]].round(3)

## 6. 읽는 법 → 3단계
- `summary_all.csv` 한 표: 전체(test_AP·F1·recall@1%·unmatched@1%) 와 허브(hub50_F1·hub100_F1). ± 는 시드 3개 표준편차 — 그 안의 차이는 무시
- 트리 vs GNN: Small 에서도 XGB 가 GNN 전부를 이기면 "이웃 구조는 이 피처 셋 위에서 추가 정보가 없다" 가 결론이고 3단계는 스태킹 vs 트리로 간다
- 스태킹 > max(GNN, 트리) 면 둘이 다른 것을 보고 있다는 뜻 — `pattern_gnn_minus_xgb.png` 에서 어느 패턴을 GNN 이 더 잡는지 확인
- 허브 F1 이 SALT 논문 주장대로 SALT·PNA 에서 높으면, 서비스에서 in-degree 높은 계좌 구간만 그 모델로 보는 2단 구성도 후보
- 3단계: `rank_mean` 상위 2개를 Large(1단계 분할, 06 데이터 파이프라인) 에서 각 1회 → report.md